# Experimentos: MLP no MNIST

Após validar a implementação da rede neural no problema XOR, o próximo passo é aplicá-la a um conjunto de dados mais complexo: o MNIST.

O MNIST é um dos datasets mais utilizados para estudos de aprendizado de máquina e visão computacional. Ele contém imagens de dígitos manuscritos de 0 a 9, representados por imagens em escala de cinza com resolução de 28 × 28 pixels.

O conjunto de dados é composto por 60.000 exemplos para treinamento e 10.000 exemplos para teste, permitindo avaliar a capacidade de generalização do modelo em um problema de classificação multiclasse.

Embora os princípios fundamentais permaneçam os mesmos utilizados no XOR (forward pass, cálculo da função de perda, backpropagation e atualização dos pesos) serão necessárias algumas adaptações para lidar com o maior volume de dados e com a classificação de múltiplas classes.

Nas etapas a seguir serão apresentadas essas modificações e sua implementação.

## Etapa 1: Carregar o MNIST

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Carrega o MNIST
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("Treino:", X_train.shape, y_train.shape)
print("Teste: ", X_test.shape, y_test.shape)
print("Valores de pixel: min =", X_train.min(), "max =", X_train.max())

I0000 00:00:1781056007.521070   68371 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781056007.525660   68371 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781056007.662699   68371 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781056010.158236   68371 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Treino: (60000, 28, 28) (60000,)
Teste:  (10000, 28, 28) (10000,)
Valores de pixel: min = 0 max = 255


## Etapa 2: Pré-processamento

Antes do treinamento da rede neural, é necessário realizar algumas transformações nos dados para adequá-los à arquitetura da MLP.

Os dados brutos apresentam dois desafios principais:

1. **Shape:** as imagens possuem dimensão `(28, 28)`, mas a rede recebe vetores como entrada. Por isso, cada imagem precisa ser convertida para um vetor de 784 atributos (`28 × 28 = 784`).

2. **Escala:** os valores dos pixels variam de 0 a 255. Para facilitar o treinamento e melhorar a estabilidade dos cálculos, os dados são normalizados para o intervalo de 0 a 1.

Além disso, os rótulos são convertidos para o formato **One-Hot Encoding**. Nesse formato, cada dígito é representado por um vetor de tamanho 10 contendo valor 1 apenas na posição correspondente à classe correta.

Por exemplo:

`3 → [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`

Essa representação será utilizada posteriormente no cálculo da função de perda para classificação multiclasse.

In [4]:
# 1. Achatar: (60000, 28, 28) → (60000, 784)
X_train_flat = X_train.reshape(60000, -1)
X_test_flat  = X_test.reshape(10000, -1)

# 2. Normalizar: 0-255 → 0-1
X_train_flat = X_train_flat / 255.0
X_test_flat  = X_test_flat  / 255.0

# 3. One-hot encoding dos labels
def one_hot(y, num_classes=10):
    n = len(y)
    oh = np.zeros((n, num_classes))
    oh[np.arange(n), y] = 1
    return oh

y_train_oh = one_hot(y_train)
y_test_oh  = one_hot(y_test)

print("X_train:", X_train_flat.shape)
print("X_test: ", X_test_flat.shape)
print("y_train:", y_train_oh.shape)
print("\nExemplo label original:", y_train[0])
print("Exemplo one-hot:       ", y_train_oh[0])

X_train: (60000, 784)
X_test:  (10000, 784)
y_train: (60000, 10)

Exemplo label original: 5
Exemplo one-hot:        [0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
